In [ ]:
import os
import re
import csv
from difflib import SequenceMatcher
from concurrent.futures import ProcessPoolExecutor
from PIL import Image
import pytesseract

In [ ]:
# Directory
BASE_DIR = os.path.expanduser('~/Desktop/extract_usernames')
SCREENSHOTS_DIR = os.path.join(BASE_DIR, 'screenshots')
OUTPUT_CSV = os.path.join(BASE_DIR, 'username_overlap.csv')

# Constants
ALLOWED_CHARS = re.compile(r'[^a-z0-9._]')
ELLIPSIS = re.compile(r'\.{2,}')

# Clean usernames
def clean_username(raw):
    text = raw.strip().lower().replace('…', '...')
    text = ELLIPSIS.sub('', text)
    return ALLOWED_CHARS.sub('', text)

def best_line(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return ''
    def score(line):
        return sum(ch.isalnum() or ch in '._' for ch in line)
    return max(lines, key=score)

# Image preprocessing and OCR extraction
PSM_FALLBACK_ORDER = [6, 7, 11]

def extract_cell_text(cell_img):
    width, height = cell_img.size
    label = cell_img.crop((0, int(height * 0.72), width, int(height * 1.0))).convert('L')
    label = label.resize((label.width * 4, label.height * 4), Image.LANCZOS)
    for psm in PSM_FALLBACK_ORDER:
        raw = pytesseract.image_to_string(label, config=f'--psm {psm}')
        text = best_line(raw)
        core = clean_username(text)
        if len(core) >= 2:
            return core
    return ''

# Screenshot segmentation and directory traversal
def screenshot_segmentation(job):
    template, path = job
    im = Image.open(path)
    width, height = im.size
    cell_w, cell_h = width / 3, height / 3
    results = []
    for row in range(3):
        for col in range(3):
            box = (int(col * cell_w), int(row * cell_h), int((col + 1) * cell_w), int((row + 1) * cell_h))
            cell = im.crop(box)
            core = extract_cell_text(cell)
            if core:
                results.append((template, core))
    return results

def collect_jobs():
    jobs = []
    for template in sorted(os.listdir(SCREENSHOTS_DIR)):
        template_dir = os.path.join(SCREENSHOTS_DIR, template)
        if not os.path.isdir(template_dir):
            continue
        for name in sorted(os.listdir(template_dir)):
            if name.endswith('.PNG'):
                jobs.append((template, os.path.join(template_dir, name)))
    return jobs

# Username canonicalisation and clustering
def similar(a, b):
    return SequenceMatcher(None, a, b).ratio()

def matches(core, canonical):
    return similar(core, canonical) >= 0.85

def build_matched_usernames(records):
    ordered = sorted(records, key=lambda r: -len(r[1]))
    matched_usernames = []

    for template, core in ordered:
        placed_username = None

        for matched_username in matched_usernames:
            if matches(core, matched_username['canonical']):
                placed_username = matched_username
                break

        if placed_username is None:
            placed_username = {
                'canonical': core,
                'templates': set()
            }
            matched_usernames.append(placed_username)

        placed_username['templates'].add(template)

    return matched_usernames

# csv export and sorting
def write_csv(matched_usernames):
    rows = []
    for matched_username in matched_usernames:
        templates_sorted = sorted(matched_username['templates'])
        rows.append((matched_username['canonical'], len(templates_sorted), '; '.join(templates_sorted)))
    rows.sort(key=lambda r: (-r[1], r[0]))
    with open(OUTPUT_CSV, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['username', 'num_templates', 'templates'])
        writer.writerows(rows)

# Main execution
def main():
    jobs = collect_jobs()
    records = []
    with ProcessPoolExecutor() as executor:
        for result in executor.map(screenshot_segmentation, jobs):
            records.extend(result)
    matched_usernames = build_matched_usernames(records)
    write_csv(matched_usernames)


if __name__ == '__main__':
    main()